In [8]:
import pandas as pd
import numpy as np
import time

start = time.time()

DATA_PATH = "../data/raw/"
df_orders = pd.read_csv(DATA_PATH + "olist_orders_dataset.csv")
df_customers = pd.read_csv(DATA_PATH + "olist_customers_dataset.csv")
df_products = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
df_order_items = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
df_payments = pd.read_csv(DATA_PATH + "olist_order_payments_dataset.csv")
df_reviews = pd.read_csv(DATA_PATH + "olist_order_reviews_dataset.csv")
df_category = pd.read_csv(DATA_PATH + "product_category_name_translation.csv")

print(f"✅ Raw data loaded in {time.time()-start:.2f} seconds")

✅ Raw data loaded in 1.72 seconds


In [9]:
# Cell 2 — Check order_status before filtering
print(df_orders['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [10]:
# Cell 3 — Filter to delivered orders, convert dates
df_orders_clean = df_orders[df_orders['order_status'] == 'delivered'].copy()

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df_orders_clean[col] = pd.to_datetime(df_orders_clean[col], errors='coerce')

print(f"Orders before filter: {len(df_orders):,}")
print(f"Orders after filter (delivered only): {len(df_orders_clean):,}")
print(f"Removed: {len(df_orders) - len(df_orders_clean):,} non-delivered orders")
print()
df_orders_clean[date_cols].dtypes

Orders before filter: 99,441
Orders after filter (delivered only): 96,478
Removed: 2,963 non-delivered orders



order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [11]:
# Cell 4 — Create derived time columns (these power your analysis later)
df_orders_clean['delivery_days'] = (
    df_orders_clean['order_delivered_customer_date'] -
    df_orders_clean['order_purchase_timestamp']
).dt.days

df_orders_clean['delivery_delay_days'] = (
    df_orders_clean['order_delivered_customer_date'] -
    df_orders_clean['order_estimated_delivery_date']
).dt.days
# Positive = delivered LATE, Negative = delivered EARLY

df_orders_clean['order_month'] = df_orders_clean['order_purchase_timestamp'].dt.to_period('M')
df_orders_clean['order_year'] = df_orders_clean['order_purchase_timestamp'].dt.year

print(df_orders_clean[['delivery_days','delivery_delay_days']].describe())

       delivery_days  delivery_delay_days
count   96470.000000         96470.000000
mean       12.093604           -11.875889
std         9.551380            10.182105
min         0.000000          -147.000000
25%         6.000000           -17.000000
50%        10.000000           -12.000000
75%        15.000000            -7.000000
max       209.000000           188.000000


In [12]:
# Cell 5 — Clean products + merge English category names
df_products_clean = df_products.copy()

# Fill missing categories
df_products_clean['product_category_name'] = (
    df_products_clean['product_category_name'].fillna('unknown')
)

# Merge English translations
df_products_clean = pd.merge(
    df_products_clean,
    df_category,
    on='product_category_name',
    how='left'
)

# For categories with no English translation (including 'unknown'), keep original
df_products_clean['product_category_name_english'] = (
    df_products_clean['product_category_name_english']
    .fillna(df_products_clean['product_category_name'])
)

print(f"Products with category: {df_products_clean['product_category_name_english'].notnull().sum():,}")
print(f"\nTop 5 categories:")
print(df_products_clean['product_category_name_english'].value_counts().head())

Products with category: 32,951

Top 5 categories:
product_category_name_english
bed_bath_table     3029
sports_leisure     2867
furniture_decor    2657
health_beauty      2444
housewares         2335
Name: count, dtype: int64


In [15]:
# Cell 6 — Build the full master table
df_main = pd.merge(df_orders_clean, df_customers, on='customer_id', how='left')
df_main = pd.merge(df_main, df_order_items, on='order_id', how='left')
df_main = pd.merge(
    df_main,
    df_products_clean[['product_id','product_category_name_english','product_weight_g']],
    on='product_id', how='left'
)

# Aggregate payments per order (remember: installments = multiple rows)
df_payments_agg = df_payments.groupby('order_id').agg(
    total_payment_value=('payment_value','sum'),
    payment_type=('payment_type','first'),
    installments=('payment_installments','max')
).reset_index()

df_main = pd.merge(df_main, df_payments_agg, on='order_id', how='left')

# Aggregate reviews per order (in case of duplicates)
df_reviews_agg = df_reviews.groupby('order_id').agg(
    review_score=('review_score','mean')
).reset_index()

df_main = pd.merge(df_main, df_reviews_agg, on='order_id', how='left')

print(f"Final df_main shape: {df_main.shape}")
print(f"Columns: {df_main.columns.tolist()}")

Final df_main shape: (110197, 28)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'delivery_delay_days', 'order_month', 'order_year', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name_english', 'product_weight_g', 'total_payment_value', 'payment_type', 'installments', 'review_score']


In [16]:
# Cell 7 — Save cleaned files
df_main.to_csv('../data/cleaned/df_main.csv', index=False)
df_orders_clean.to_csv('../data/cleaned/orders_clean.csv', index=False)
df_products_clean.to_csv('../data/cleaned/products_clean.csv', index=False)

print("Saved 3 cleaned files to data/cleaned/")
print(f"df_main.csv: {df_main.shape[0]:,} rows, {df_main.shape[1]} columns")

Saved 3 cleaned files to data/cleaned/
df_main.csv: 110,197 rows, 28 columns


## Day 3 — Data Cleaning Decisions Log

1. FILTERED orders to order_status == 'delivered' only
   → Removed 2,963 non-delivered orders (cancelled, processing, etc.)
   → Reason: only completed transactions represent real business

2. CONVERTED all 5 date columns from string to datetime
   → Used errors='coerce' to handle any malformed dates safely

3. CREATED derived columns: delivery_days, delivery_delay_days,
   order_month, order_year
   → These power Week 3's time-based analysis

4. FILLED 610 missing product_category_name values with 'unknown'
   → Reason: dropping would lose real revenue data;
     less than 2% of products affected

5. MERGED category_translation table for English category names
   → Fallback to Portuguese name if no translation exists

6. AGGREGATED payments per order_id (SUM payment_value)
   → Reason: orders with installments had multiple payment rows

7. AGGREGATED reviews per order_id (MEAN review_score)
   → Reason: handle any duplicate review entries safely

Output: data/cleaned/df_main.csv (110,987 rows, 28 columns)

In [17]:
print("Unique orders in df_main:", df_main['order_id'].nunique())
print("Rows in df_orders_clean:", len(df_orders_clean))

Unique orders in df_main: 96478
Rows in df_orders_clean: 96478


In [18]:
print(df_main.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'delivery_delay_days', 'order_month', 'order_year', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name_english', 'product_weight_g', 'total_payment_value', 'payment_type', 'installments', 'review_score']


In [21]:
print("Nulls in price:", df_main['price'].isnull().sum())
print("Nulls in total_payment_value:", df_main['total_payment_value'].isnull().sum())
print("Nulls in review_score:", df_main['review_score'].isnull().sum())

Nulls in price: 0
Nulls in total_payment_value: 3
Nulls in review_score: 827


In [22]:
df_main[df_main['total_payment_value'].isnull()][['order_id','order_status','price','order_purchase_timestamp']]

,order_id,order_status,price,order_purchase_timestamp
34056,bfbd0f9bdef84302105ad712db648a6c,delivered,44.99,2016-09-15 12:16:38
34057,bfbd0f9bdef84302105ad712db648a6c,delivered,44.99,2016-09-15 12:16:38
34058,bfbd0f9bdef84302105ad712db648a6c,delivered,44.99,2016-09-15 12:16:38


In [23]:
nulls_df = df_main[df_main['total_payment_value'].isnull()]
print("Affected rows:", len(nulls_df))
print("Affected unique orders:", nulls_df['order_id'].nunique())

Affected rows: 3
Affected unique orders: 1


## Day 3 — Data Cleaning Decisions Log

1. FILTERED orders to order_status == 'delivered' only
   → Removed 2,963 non-delivered orders (cancelled, processing, etc.)
   → Reason: only completed transactions represent real business

2. CONVERTED all 5 date columns from string to datetime
   → Used errors='coerce' to handle any malformed dates safely

3. CREATED derived columns: delivery_days, delivery_delay_days,
   order_month, order_year
   → These power Week 3's time-based analysis

4. FILLED 610 missing product_category_name values with 'unknown'
   → Reason: dropping would lose real revenue data;
     less than 2% of products affected

5. MERGED category_translation table for English category names
   → Fallback to Portuguese name if no translation exists

6. AGGREGATED payments per order_id (SUM payment_value)
   → Reason: orders with installments had multiple payment rows

7. AGGREGATED reviews per order_id (MEAN review_score)
   → Reason: handle any duplicate review entries safely

8. INVESTIGATED 3 NULLs in total_payment_value after merge
   → Found all 3 rows belonged to ONE order (3 items, same order_id)
   → Confirmed independently in raw payments.csv: this order has 
     no payment record at all — a genuine gap in Olist's source data
   → Decision: left as NaN rather than fabricating a payment value
   → Affects 1 order out of 96,478 — negligible impact

9. INVESTIGATED 827 NULLs in review_score
   → Confirmed this is a real business fact, not a data error:
     these customers placed an order but left no rating at all
   → Decision: left as NaN, will be handled explicitly in Week 3 
     review analysis (e.g. "do non-reviewers behave differently?")

Output: data/cleaned/df_main.csv (110,197 rows, 28 columns)
Verified: df_main['order_id'].nunique() == len(df_orders_clean) == 96,478